# Problem Set 1: Machine Learning in Economics

This notebook combines supervised and unsupervised learning in four applications. Baseline code is supplied: run it, interpret the outputs, and use it as a starting point for the extensions. Familiarity with introductory statistics, regression, and the machine learning concepts from the lectures is sufficient for the guided questions; the opening examples introduce the notebook workflow. 

## Getting Started

- **Code cells** contain Python instructions. **Markdown cells** contain explanations, equations, and your written answers.
- Run a selected cell with **Shift + Enter**. Run cells in order: later cells depend on variables created earlier.
- The **kernel** is the Python process that remembers variables. Restarting it clears that memory. Deleting an output does not delete the variables.
- After changing an earlier cell, rerun the cells that depend on it. Before submitting, restart the kernel and run all cells from top to bottom.
- An error message stops that cell. Read its final line, check the variable names and earlier cells, and rerun after correcting the issue. A cell marked as running may still be downloading data or fitting models.
- Save regularly. Answer each question in its following Markdown cell; place any extension code immediately after that answer.

### Packages and Imports

A **package** provides reusable tools. Installing a package makes it available to the Python environment; importing it makes its functions available in this session.

| Package | Purpose in this notebook |
|---|---|
| NumPy (`np`) | Numerical arrays and calculations |
| pandas (`pd`) | Data tables, dates, and data preparation |
| matplotlib (`plt`) and seaborn (`sns`) | Plots |
| scikit-learn (`sklearn`) | Models, preprocessing, and evaluation |

These packages are generally available in Google Colab. In another Jupyter environment, if an import fails with `ModuleNotFoundError`, uncomment and run the installation cell below, then retry the import. `%pip` installs into the notebook's Python environment.

The data cells download three CSV files automatically and require internet access. The equity file is approximately 79 MB and is cached in the notebook's working folder for reuse. The files are also available on OLAT. If you have any problems with running this notebook, just write the TA an email.


In [ ]:
# Run only if the required packages are missing.
# %pip install numpy pandas matplotlib seaborn scikit-learn


### A Short Python Warm-up

`=` assigns a value to a name. `#` starts a comment. Lists store several values; a pandas `DataFrame` is a table whose columns can be selected by name. Parentheses call functions; a dot accesses a function or attribute attached to an object. Python names are case-sensitive, and indentation groups instructions in loops and functions.

Run the example and try changing one income value. This warm-up is ungraded.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

incomes = [30_000, 45_000, 60_000]
example = pd.DataFrame({"income": incomes, "purchase": [0, 1, 1]})
display(example)
print("Mean income:", example["income"].mean())
print("Purchase rate:", example["purchase"].mean())


### How to Work Through the Exercises

For each part, run the supplied code and explain the outputs in their economic context. For the extensions, modify the baseline code, compare alternatives, and justify your choices.

The extensions are an opportunity to explore questions using your own code. Don’t worry if some questions seem difficult or if your changes do not improve performance. To receive credit, show that you have made a thoughtful attempt and explain what you learned. The focus is on the learning process rather than a perfect outcome.

Write all your answers directly in this Jupyter notebook, in the spaces provided.

Some variable names are reused between parts. To rerun a part after working on another, rerun all its code cells from the start of that part.

1. **Predictability of equity returns:** supervised regression.
2. **Predicting household purchases:** classification.
3. **Common variation in equity returns:** PCA and clustering.
4. **Household heterogeneity:** representation and clustering.

Have fun working through this homework!

## Part 1. Predictability of Equity Returns

### Economic context
Can past returns and recent market conditions help predict daily equity returns? Does predictability differ when we study the magnitude of returns rather than their direction?

You will use a panel of daily equity returns, select one ticker, and compare prediction rules. The emphasis is on out-of-sample evidence, information available at the forecast date, and interpretation of financial risk.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score


## Constructing the Return Matrix

Before we can build any predictive model, we need to put the return data into a form that is usable for time-series machine learning.

In the raw data, returns typically appear in a “long” format: Each row corresponds to a particular date and a particular asset (ticker), along with that asset's return on that date. While this format is convenient for storage, it is not the most convenient format for modeling.

We thus start by constructing a **return matrix**.



In [ ]:
#Config

startdate = '2011-01-01' # This can be updated as needed ( Minimum date is 2011-01-01 as per the data available )
enddate = '2024-12-31'  # This can be updated as needed ( Maximum date is 2024-12-31 as per the data available )

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

EQUITY_URL = "https://www.dropbox.com/scl/fi/glm90k03u7yehadyw9dk5/df_factor.csv?rlkey=iwhnsi7sf9ogsvfpm9jltqzew&dl=1"
equity_file = Path("df_factor.csv")
if not equity_file.exists():
    urlretrieve(EQUITY_URL, equity_file)

df = pd.read_csv(equity_file, parse_dates=["date"])
tic_list = df.loc[df["date"] < startdate, "tic"].unique()
df = df[df["tic"].isin(tic_list[:100])].copy()
df


In [ ]:
df = df[['date', 'tic', 'ret']]

In [ ]:
df.sort_values('date', inplace=True)

In [ ]:
returns_matrix = df.pivot(index='date', columns='tic', values='ret')
returns_matrix = returns_matrix.dropna()

In [ ]:
returns_matrix = returns_matrix.loc[startdate:enddate]

In [ ]:
returns_matrix

## Set-up Questions

---

**Question 1.** Provide a brief explanation of what the code blocks above are doing. Importantly, what does the `pivot` step accomplish? After this step, what does each row represent and what does each column represent?

What does the `dropna` step accomplish? Why is this step potentially important? Does the presence of this step raise any concerns? Explain.

---

*Your answer:*



Here we look a little more at the data. We specifically look at the asset `AAPL`. You may choose to alter this to a different ticker (or tickers) if you wish. The tickers for the assets we have data on are provided by executing the first code block below.

In [ ]:
returns_matrix.columns

In [ ]:
target_asset = 'AAPL' # This can be updated as needed

returns = returns_matrix[target_asset].copy()

# Summary statistics
returns.describe()

In [ ]:
# Distribution of returns
returns.hist(bins=100)
plt.title("Distribution of Daily Returns")
plt.show()


**Question 2.** Examine the histogram of daily returns for your chosen asset. What features stand out? What do these features imply for financial risk and the evaluation of forecasts?


*Your answer:*



## Feature Construction: Autoregressive Predictors

We now move from raw returns to predictive features.

In this section, we construct a simple autoregressive feature set. The idea is straightforward: Past returns may contain information about future returns. By lagging the return series, we create predictors that allow the model to learn temporal patterns.

This setup corresponds to a linear autoregressive model of the form:

$$
r_{t+1} = \beta_0 + \beta_1 r_t + \beta_2 r_{t-1} + \cdots + \beta_k r_{t-k} + \varepsilon_{t+1}.
$$

It is important to think carefully about alignment. Predictors must use only information available at time t to predict returns at time t+1. Any leakage of future information would invalidate the model evaluation.


In [ ]:
data = pd.DataFrame({
    "y": returns
})

for lag in range(1,6):
    data[f"lag_{lag}"] = returns.shift(lag)

data.dropna(inplace=True)
data.head()


We can also create a few rolling window features like taking the mean return value of the last 30 days or the standard deviation of returns for the last 30 days

In [ ]:
data["ma_30"] = returns.shift(1).rolling(30).mean()
data["vol_30"] = returns.shift(1).rolling(30).std()

data.dropna(inplace=True)


---

**Question 3.** What is the `.shift(1).rolling(30).mean()` doing? Why is the `shift(1)` important for building a predictive model?

---

*Your answer:*



We can also incorporate information from other assets. Here, we're taking a particularly simplistic approach.

In [ ]:
market_ret = returns_matrix.mean(axis=1)

data["market_avg"] = market_ret.shift(1)
data["market_avg"].describe()

---

**Question 4.** What does the variable `market_avg` capture?

---

*Your answer:*



## Baseline Predictive Models

We now move from feature construction to model estimation.

The baseline model is a simple linear regression that uses the autoregressive features to predict next-period returns. This corresponds to a classical time-series forecasting setup: estimate parameters using historical data and evaluate performance out-of-sample.

The workflow in this section follows a standard train–test framework:

First, the dataset is split into a training period and a testing period. The model is estimated using only the training sample. Once the parameters are fixed, predictions are generated for the testing sample.

We will be building two baseline models here. One which simply uses the previous period's return to predict the next day's return and the other which predicts returns using the 30 day moving average.

In [ ]:
split_date = data.index[-252]

train = data.loc[data.index < split_date].copy()
test = data.loc[data.index >= split_date].copy()
assert train.index.intersection(test.index).empty

X_train = train.drop("y", axis=1)
y_train = train["y"]
X_test = test.drop("y", axis=1)
y_test = test["y"]


---

**Question 5.** In time series settings, random train/test splits can be misleading. Explain why randomly selecting the train/test split could be problematic in this return forecasting context. Give two sensible alternative splitting/validation approaches beyond the simple "hold out the last year" approach taken above.

(Think about how the model might be deployed, evaluated, and updated in practice and how this might be represented by the train/test setup.)

---

*Your answer:*



## Simple baseline models

In [ ]:
y_pred_naive = X_test["lag_1"]


In [ ]:
y_pred_ma = test["ma_30"]


To evaluate our models, we will be using the Mean Squared Error metric.

In [ ]:
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return mse

print("Naive:", evaluate(y_test, y_pred_naive))
print("MA30:", evaluate(y_test, y_pred_ma))


In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

evaluate(y_test, y_pred_lr)


In [ ]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

lasso = Lasso(alpha=0.001)
lasso.fit(X_train, y_train)

print('Ridge Regression:', evaluate(y_test, ridge.predict(X_test)))
print('Lasso Regression:', evaluate(y_test, lasso.predict(X_test)))

---

**Question 6.** Provide a brief explanation of each of the baseline models. Which of the baseline models performs better according to our MSE criterion? How might one interpret the MSEs - e.g., what do these numbers tell us about the quality of the forecasts? (HINT: You might refer back to the summary statistics provided above.)

---

*Your answer:*



### Extension: Returns and Absolute Returns

Build two prediction rules for the asset studied above: one for returns and one for absolute returns. Absolute returns measure the magnitude of daily movements; their conditional expectation is distinct from conditional variance. There is no single correct model, and finding little improvement over a simple baseline is a valid result.

**Question 7.** Build and compare models beyond the supplied baselines for both targets. You may construct additional features from the chosen asset or other assets in the dataset. Explain:

- the features you use and why they may contain predictive information;
- how you avoid look-ahead bias and data leakage;
- how you choose tuning parameters using validation data or time-series cross-validation within the training sample;
- which evaluation metrics you use for each target and why;
- which models the evidence supports and what you learn about return predictability.

For absolute returns, construct the target using `.abs()` and fit separate models. Do not use the absolute return being predicted as an input. The displayed test results above illustrate evaluation; do not use that sample to tune your extensions.


*Your answer:*



## Part 2. Predicting Household Purchase Decisions

### Economic context
Households face a price for a premium dishwasher detergent and decide whether to purchase. Prices vary with input costs, supply conditions, macroeconomic conditions, promotions, and pricing decisions.

The data are simulated household-period observations with heterogeneous characteristics and time-varying conditions. Your task is to predict purchases conditional on observable characteristics and prices, and assess how different prediction rules perform.

This is a prediction exercise. Predictive associations with price do not, by themselves, identify the causal effect of changing prices.


## Data

### Observational Unit

Each row in the dataset represents a specific household in a specific period observing a specific price.

### Target (Outcome) Variable

- **`purchase`**  
  = 1 if the household purchased the detergent in that period  
  = 0 otherwise  

### Key Explanatory Variables

- **`price`**  
  The price of the product in that period.

- **`income`**  
  Household income.

- **`age`**  
  Age of the household head.

- **`family_size`**  
  Number of people in the household.

- **`prev_interactions`**  
  Number of prior interactions with the brand

- **`state`**  
  U.S. state of residence.

- **`sex`**  
  Sex of household head.

- **`race`**  
  Race category of household head.

- **`period`**  
  Time index (captures macroeconomic evolution and firm pricing adjustments).

- **`hh_id`**  
  Household identifier (used only for tracking, not prediction).


In [ ]:
#Libraries for this case
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, log_loss, confusion_matrix
)

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree

np.random.seed(42)


## Set-up Questions

---

**Question 1.** Briefly explain what the import cell above is doing.

- Why do we import libraries before doing anything else?
- What is the purpose of setting a random seed with `np.random.seed(42)`?

---

*Your answer:*



In [ ]:
# Load CSV from public Dropbox link
DROPBOX_CSV_URL = "https://www.dropbox.com/scl/fi/dsbjrgur2orirwsvolmm7/pricing_application_hh_empirical.csv?rlkey=xlt4vba8bn6y8qt6m59zxxjwp&dl=1"

def load_csv(url: str) -> pd.DataFrame:
    df = pd.read_csv(url)
    return df

df = load_csv(DROPBOX_CSV_URL)
df.describe()


We predict `purchase` using numerical and categorical features. We exclude `hh_id` to avoid treating an identifier as a substantive characteristic. The baseline includes `period`; you may compare results with and without it. Whether that variable is useful depends on which observations you intend to predict.


In [ ]:
# 1. Drop common index-like columns if present
for col in ["Unnamed: 0", "index", "Index"]:
    if col in df.columns:
        df = df.drop(columns=[col])

# 2. Ensure target exists
TARGET = "purchase"
assert TARGET in df.columns, f"Expected target column `{TARGET}` not found."

# 3. Ensure target is binary integer
df[TARGET] = df[TARGET].astype(int)

# 4. Report dataset dimensions
print(f"Number of observations: {df.shape[0]}")
print(f"Number of variables: {df.shape[1]}")

# 5. Define exclusions (avoid ID leakage)
EXCLUDE = [TARGET]
if "hh_id" in df.columns:
    EXCLUDE.append("hh_id")

# 6. Create X and y
X = df.drop(columns=[c for c in EXCLUDE if c in df.columns]).copy()
y = df[TARGET].copy()

# 7. Detect feature types automatically
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()

print("\nNumeric columns:", num_cols)
print("Categorical columns:", cat_cols)
print("\nFeature matrix shape:", X.shape)
print("Target mean (purchase rate):", y.mean())


**Question 2.** Using the summary statistics:

1. Would always predicting “no purchase” achieve high accuracy? What does this imply for the evaluation metrics you emphasize?
2. Describe the variation in `price`. Why does price variation matter for studying purchase behavior? Distinguish predictive relationships from causal price effects.


*Your answer:*



---

**Question 3.** Why do we explicitly remove identifier-like variables such as `hh_id` from the feature matrix? What could go wrong if we allowed the model to use household IDs for prediction?

Why do we separate the dataset into `X` (features) and `y` (target) before modeling? What conceptual mistake would we make if `purchase` were accidentally included in `X`?

---   

*Your answer:*



### Randomization and splits (train/validation/test)

In [ ]:
# Split 1: Train/test split WITHOUT shuffling.
# This keeps the original row order
X_train_ns, X_test_ns, y_train_ns, y_test_ns = train_test_split(
    X, y, test_size=0.25, shuffle=False
)

# Split 2: Train/test split WITH shuffling + stratification.
# - shuffle=True randomizes the row order before splitting, so train/test are more comparable.
# - random_state=42 makes the split reproducible (same split each time you run).
# - stratify=y preserves the overall purchase rate (class balance) in both train and test sets.
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X, y, test_size=0.25, shuffle=True, random_state=42, stratify=y
)

# Helper function to summarize a split by checking whether the class balance
# is satisfied
def summarize(y_tr, y_te, name):
    print(f"\n{name}")
    print("Train purchase rate:", y_tr.mean(), "| n=", len(y_tr))
    print("Test  purchase rate:", y_te.mean(), "| n=", len(y_te))

summarize(y_train_ns, y_test_ns, "No shuffle")
summarize(y_train_s, y_test_s, "Shuffle + stratify")


In [ ]:
# Use the shuffled split as standard
X_train_full, X_test, y_train_full, y_test = X_train_s, X_test_s, y_train_s, y_test_s

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=42, stratify=y_train_full
)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)
print("Purchasing rates:", y_train.mean(), y_val.mean(), y_test.mean())


---

**Question 4.** Why do we create separate train/validation/test splits. In the data, there is a `period` variable. Here, we are randomizing by independently sampling ignoring the presence of this variable. Explain circumstances where this splitting regime is sensible and provide an example of a circumstance where it may be a bad idea.

---

*Your answer:*



### Baseline model: Logistic regression

In [ ]:
# ColumnTransformer allows us to apply different preprocessing
# steps to different types of variables (numeric vs categorical).

preprocess = ColumnTransformer(
    transformers=[
        # 1. Standardize numeric variables:
        #    - Centers them at mean 0
        #    - Scales them to standard deviation 1
        ("num", StandardScaler(), num_cols),
        # 2. One-hot encode categorical variables
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop"
)
# Logistic regression model
logit = LogisticRegression(max_iter=3000)

# This ensures:
#   - Preprocessing is fit only on training data
#   - The exact same transformations are applied to validation/test data
#   - No data leakage occurs
pipe_logit = Pipeline([
    ("prep", preprocess),
    ("model", logit)
])
pipe_logit.fit(X_train, y_train)


In [ ]:
def eval_classifier(pipe, X, y, name="set"):
    """
    Evaluate a fitted classification pipeline on a given dataset.

    Parameters:
        pipe : fitted sklearn Pipeline
        X    : feature matrix
        y    : true labels (0/1 purchase)
        name : label identifying the dataset (train/val/test)

    Returns:
        Dictionary of performance metrics.
    """
    # 1. Predict purchase probabilities.
    p = pipe.predict_proba(X)[:, 1]
    # 2. Assume individuals will purchase the product if probability >= 0.5.
    # You can modify this threshold.
    yhat = (p >= 0.5).astype(int)

    # 2. Compute evaluation metrics
    out = {
        "set": name,
        "accuracy": accuracy_score(y, yhat),                        # Fraction of correct predictions.
        "precision": precision_score(y, yhat, zero_division=0),     # Precision: Of predicted purchases, how many were correct?
        "recall": recall_score(y, yhat, zero_division=0),           # Recall: Of actual purchases, how many did we correctly identify?
        "f1": f1_score(y, yhat, zero_division=0),                   # F1: Harmonic mean of precision and recall
        "roc_auc": roc_auc_score(y, p),                             # ROC AUC: Measures ranking quality (threshold-independent).
        "log_loss": log_loss(y, p),                                 # Log loss: Measures quality of predicted probabilities.
    }
    return out

res_train = eval_classifier(pipe_logit, X_train, y_train, "train")
res_val   = eval_classifier(pipe_logit, X_val, y_val, "val")
res_test  = eval_classifier(pipe_logit, X_test, y_test, "test")

pd.DataFrame([res_train, res_val, res_test])


### Interpreting Purchase Predictions

The next cells construct a cumulative gain chart and inspect logistic regression coefficients. The chart describes how observed purchases are concentrated among observations ranked by predicted probability. It does not measure the causal effect of contacting households or changing prices.


In [ ]:
# 1. Get predicted probabilities on test set
p_test = pipe_logit.predict_proba(X_test)[:, 1]

# 2. Create dataframe for sorting
gain_df = (
    pd.DataFrame({
        "y_true": y_test.values,
        "p_hat": p_test
    })
    .sort_values("p_hat", ascending=False)
    .reset_index(drop=True)
)

# 3. Compute cumulative metrics
gain_df["cum_samples"] = np.arange(1, len(gain_df) + 1) / len(gain_df)
gain_df["cum_positives"] = gain_df["y_true"].cumsum()
total_positives = gain_df["y_true"].sum()
gain_df["cum_gain"] = gain_df["cum_positives"] / total_positives

# 4. Plot cumulative gain curve
plt.figure(figsize=(8,6))
plt.plot(gain_df["cum_samples"], gain_df["cum_gain"], label="Logistic Model")
plt.plot([0,1], [0,1], linestyle="--", label="Random Baseline")

plt.xlabel("Fraction of Observations Selected")
plt.ylabel("Fraction of Observed Purchases Included")
plt.title("Cumulative Gain Chart (Test Set)")
plt.legend()
plt.grid(True)
plt.show()

# 5. Summary for the top 20% of ranked observations
top_20_index = max(1, int(0.20 * len(gain_df))) - 1
top_20_capture = gain_df.loc[top_20_index, "cum_gain"]

print(f"The top 20% of observations ranked by predicted probability")
print(f"include approximately {top_20_capture:.2%} of all purchases.")

In [ ]:
# --- Simple interpretation: largest logistic regression coefficients ---

# Extract feature names after preprocessing
feat_names = pipe_logit.named_steps["prep"].get_feature_names_out()
coefs = pipe_logit.named_steps["model"].coef_.ravel()

coef_df = pd.DataFrame({"feature": feat_names, "coef": coefs})
coef_df = coef_df.sort_values("coef")

print("Most negative coefficients (lower purchase propensity):")
display(coef_df.head(10))

print("\nMost positive coefficients (higher purchase propensity):")
display(coef_df.tail(10))


**Question 5.** Use the logistic regression baseline and gain chart.

- Explain whether ranking observations by predicted probability identifies more observed purchases than selecting observations at random.
- Why might a higher overall ROC AUC fail to imply better performance within the top 20% of ranked observations?
- Which variables have the largest coefficients? Interpret numerical coefficients in light of standardization and categorical coefficients in light of their encoding. Why should these coefficients not automatically be interpreted as causal effects?


*Your answer:*



### Extension: Comparing Purchase Prediction Rules

**Question 6.** Build and compare at least **two** additional prediction rules beyond the supplied logistic regression baseline, such as decision trees, random forests, gradient boosting, or k-nearest neighbors.

Explain preprocessing, hyperparameter tuning using the validation set or suitable cross-validation within the training data, your evaluation metrics, and your final model choice. The logistic regression baseline already uses regularization; changing its penalty is a tuning comparison. Focus your conclusions on out-of-sample purchase prediction and household heterogeneity.

The test metrics and gain chart above illustrate the evaluation procedure. Use validation data for exploratory comparisons and evaluate your final selected models on the test sample after fixing your choices.


*Your answer:*



## Part 3. Common Variation in Equity Returns


### Economic context

How much of the variation in equity returns is shared across assets? Can statistical components and clusters reveal economically interpretable patterns of co-movement?

You will apply principal component analysis (PCA) and clustering to the equity panel used in Part 1. Investigate common variation, diversification, and the stability of return structure over time.

PCA components are statistical objects. Economic interpretations require evidence, such as sector characteristics or external factor series. Full-sample analysis is appropriate for retrospective description; if components later become forecasting inputs, preprocessing and PCA must be fitted using only the training sample.


### Constructing the Return Matrix

We reconstruct the return matrix from Part 1 so that this part can be rerun from its beginning. The downloaded equity file is reused.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA, FactorAnalysis
import seaborn as sns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.width', 500)

In [ ]:
#Config

startdate = '2011-01-01' # Change this to your desired start date (Minimum: 2011-01-01)
enddate = '2024-12-31'  # Change this to your desired end date (Maximum: 2024-12-31)

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

EQUITY_URL = "https://www.dropbox.com/scl/fi/glm90k03u7yehadyw9dk5/df_factor.csv?rlkey=iwhnsi7sf9ogsvfpm9jltqzew&dl=1"
equity_file = Path("df_factor.csv")
if not equity_file.exists():
    urlretrieve(EQUITY_URL, equity_file)

df = pd.read_csv(equity_file, parse_dates=["date"])
tic_list = df.loc[df["date"] < startdate, "tic"].unique()
df = df[df["tic"].isin(tic_list[:100])].copy()
df


In [ ]:
df = df[['date', 'tic', 'ret']]

In [ ]:
df.sort_values('date', inplace=True)

In [ ]:
returns_matrix = df.pivot(index='date', columns='tic', values='ret')
returns_matrix = returns_matrix.dropna()
returns_matrix = returns_matrix.loc[startdate:enddate]


## Standardization

Before PCA, we typically standardize each column (ticker) so that PCA focuses on *correlation-style* co-movement rather than simply variance differences driven by a few volatile assets.


In [ ]:
returns_standardized = (returns_matrix - returns_matrix.mean()) / returns_matrix.std()  #Z score Standardization


---

**Question 1.** Why might standardization make sense when looking at unsupervised learning in a cross-section of equities? E.g., what would happen if one ticker had much higher volatility than the others.

When might you *not* want to standardize before PCA (or applying other unsupervised learning methods)?

If you wanted to use unsupervised learning as part of a supervised learning pipeline that includes train/test splits, what mistake has been made in the above code that does the standardization?

---

*Your answer:*



## Baseline PCA

We now look at the first two principal components extracted from PCA applied to the standardized return matrix.

At this stage, the choice of two components is arbitrary. It is not meant to be optimal or economically justified. Instead, this serves as a baseline specification that we will later revisit and improve.

PCA extracts directions in the data that explain the largest share of cross-sectional variance. The resulting components are purely statistical objects. Whether they correspond to economically meaningful forces is something you will need to evaluate.

From the PCA output, we construct two objects:

- The **factor loadings**, which describe how each stock loads on each extracted component.
- The **factor time series**, which describe how the extracted components evolve over time.

In [ ]:
pca = PCA(n_components=2) # n_componments = number of factors
factors = pca.fit_transform(returns_standardized)

In [ ]:
factor_loadings = pd.DataFrame(pca.components_.T,
                               index=returns_standardized.columns,
                               columns=["Factor1", "Factor2"])

factors_df = pd.DataFrame(factors, index=returns_standardized.index,
                          columns=["Factor1", "Factor2"])

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(factor_loadings, annot=False, cmap="coolwarm", center=0)
plt.title("Factor Loadings (Stock Sensitivities)")
plt.show()

---

**Question 2.** The code block above produces a heatmap showing the factor loadings. Examine the heatmap of factor loadings carefully. Do the exposures suggest an economic story (e.g., a market-wide factor, growth vs value, tech vs non. tech)? What additional data would you want to validate your story (e.g., sector labels, fundamentals, ...)?

---

*Your answer:*



In [ ]:
factors_df.plot(figsize=(12,6), title="Extracted Latent Factors")
plt.show()

**Question 3.** Inspect the time series of the two components. Compare their variability and behavior during turbulent periods. PCA scores are uncorrelated over the fitting sample by construction: does this imply independence, or prevent both components from showing large movements during a crisis?


*Your answer:*



## Exploring Factor Behavior at a More Granular Level

The next two cells allow you to inspect the extracted factors more closely.

First, you can isolate a specific year and examine how the latent factors behave within that shorter window. This can be useful for identifying regime changes, crisis dynamics, or shifts in factor dominance that may not be obvious in the full-sample plot.

Second, you can overlay the return of a specific asset on top of one or more extracted factors. This provides a more detailed view of how individual securities co-move with the latent components. While this is not a formal regression, it can help build intuition about explanatory power and exposure patterns.

In [ ]:
year = 2020 # Change this to the year you want to visualize

factors_df[factors_df.index.year == year].plot(
    figsize=(12,6),
    title=f"Extracted Latent Factors — {year}"
)

plt.show()

In [ ]:
year = 2020 # Change this to the year you want to visualize
asset = "AAPL" # Change this to the asset you want to visualize
factor_choice = "All"   # "Factor1" / "Factor2" / "All"

ymin = factors_df.loc[str(year)].min().min()
ymax = factors_df.loc[str(year)].max().max()
padding = 0.05 * (ymax - ymin)

cols = factors_df.columns if factor_choice == "All" else [factor_choice]

ax = factors_df.loc[str(year), cols].plot(
    figsize=(12,6),
    title=f"{factor_choice} + {asset} Returns — {year}"
)

ax.set_ylim(ymin - padding, ymax + padding)

returns_matrix.loc[str(year), asset].plot(
    ax=ax,
    secondary_y=True,
    label=asset,
    color="black",
    alpha=0.5,
    mark_right=False
)

ax.legend(ax.get_legend_handles_labels()[0] + ax.right_ax.get_legend_handles_labels()[0],
          ax.get_legend_handles_labels()[1] + ax.right_ax.get_legend_handles_labels()[1])

plt.show()


---

**Question 4.** Use these visualizations thoughtfully. Vary the year, the asset, and the selected factor(s).

- Does the relationship between an asset and a factor appear stable across time?
- Does one factor seem to matter more for certain assets?
- Are there periods where the factors fail to track asset behavior?

---


*Your answer:*



## Clustering: do groups of stocks behave similarly?

PCA compresses the panel into a few dimensions. We can then cluster stocks based on their factor loadings.

One simple approach:

- represent each ticker by its loadings on the first `N_FACTORS` components,
- apply a clustering algorithm (e.g., k-means),
- see whether clusters look “sensible” (potentially corresponding to industries, styles, or risk exposures).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Build the feature matrix for clustering: each ticker is represented by its loadings
X_cluster = factor_loadings.values

# Arbitrarily choose K = 4
K = 4

km = KMeans(n_clusters=K, random_state=0, n_init=50)
cluster_labels = km.fit_predict(X_cluster)

clusters = pd.Series(cluster_labels, index=factor_loadings.index, name='cluster')
print(clusters.value_counts().sort_index())

# Show a few tickers per cluster
for k in range(K):
    members = clusters[clusters == k].index.tolist()
    print(f"\nCluster {k}: n = {len(members)}")
    print(members[:20])


In [ ]:
# 2D scatter of loadings on the two factors
plt.figure(figsize=(8,6))
plt.scatter(factor_loadings.iloc[:,0], factor_loadings.iloc[:,1], c=cluster_labels)
plt.title('Clusters visualized in (loading on Factor 1, loading on Factor 2) space')
plt.xlabel('Loading on Factor 1')
plt.ylabel('Loading on Factor 2')
plt.show()



---

**Question 5.** Pick **one** cluster and investigate it: do the tickers look like they come from similar industries? (You can use the web, Bloomberg/Yahoo, or AI tools to quickly look up what the tickers are.)

 How would you validate whether the clusters are truly “industry clusters” vs “style/risk clusters” vs something more idiosyncratic?


*Your answer:*



### Extension: Structure and Stability

The baseline uses two PCA components and four clusters without justifying these choices.

**Question 6.** Extend the baseline analysis:

1. Choose and justify the number of PCA components, using diagnostics such as explained variance.
2. Use PCA outputs to investigate cross-sectional structure and apply at least one additional unsupervised method, such as clustering on component weights or correlations, hierarchical clustering, or anomaly detection.
3. Assess stability across periods, for example before and after 2020. When comparing PCA fits, remember that signs can flip and components can change order without implying an economic change.

Explain what each method optimizes, what constitutes convincing evidence of structure, and what additional data would help assess its economic interpretation.


*Your answer:*



## Part 4. Household Heterogeneity and Purchase Responses


### Economic context

A conjoint-style study asks households whether they would purchase a premium dishwasher detergent at several prices. Each household appears more than once in the raw data.

Your task is to investigate heterogeneity across households using unsupervised learning. Examine whether the data reveal interpretable groups and how the groups depend on feature construction, distance, and clustering choices. A clustering algorithm creates groups even when the underlying population does not contain sharply separated types.


### Data

Each row in the dataset represents a household/price combination. That is, each row corresponds to a household asked if they would purchase the product at a given price. Each household was asked the yes/no purchase at question at several prices, so each household appears in the raw data multiple times.

## Variables

- **`purchase`**  
  = 1 if the household is willing to purchase the detergent at that price
  = 0 otherwise  

- **`price`**  
  The price of the product.

- **`income_bracket`**  
  Household income bracket.

- **`age`**  
  Age of the household head.

- **`family_size`**  
  Number of people in the household.

- **`prev_interactions`**  
  Number of prior interactions with the brand

- **`state`**  
  U.S. state of residence.

- **`sex`**  
  Sex of household head.

- **`race`**  
  Race category of household head.

- **`household_id`**  
  Household identifier.



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import pairwise_distances
from sklearn.metrics import silhouette_score, silhouette_samples

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

np.random.seed(42)

## Loading and constructing the household-level dataset

The raw dataset contains repeated observations for households at different prices.  For clustering, it is more natural to work at the household level rather than at the household-price-row level. The code below creates a simple household-level table.  

In [ ]:
DROPBOX_CSV_URL = "https://www.dropbox.com/scl/fi/qzvgjnwdwipv4tkah6qvl/pricing_application_conjoint.csv?rlkey=l9sdcy2hw9daa3iv4491jp0je&dl=1"

df_raw = pd.read_csv(DROPBOX_CSV_URL)

for col in ["Unnamed: 0", "index", "Index"]:
    if col in df_raw.columns:
        df_raw = df_raw.drop(columns=col)

reservation_price = (
    df_raw.loc[df_raw["purchase"] == 1]
    .groupby("household_id")["price"]
    .max()
    .rename("reservation_price")
)

households = (
    df_raw.drop_duplicates("household_id")
    .merge(reservation_price, on="household_id", how="left")
)

households["reservation_price"] = households["reservation_price"].fillna(10)

households.head()


In [ ]:
exclude = ["purchase", "price"]
if "household_id" in households.columns:
    exclude.append("household_id")
if "hh_id" in households.columns:
    exclude.append("hh_id")

X = households.drop(columns=[c for c in exclude if c in households.columns]).copy()

num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()

print("Numeric columns:", num_cols)
print("Categorical columns:", cat_cols)
print("Feature matrix shape:", X.shape)


**Question 1.** Explain the construction of the household-level dataset. Why cluster households rather than household-price rows? What does `reservation_price` measure, and what limits its interpretation as willingness to pay?

Under monotone, deterministic responses, the highest accepted tested price is a lower bound on willingness to pay. Always-buyers and never-buyers are censored at opposite ends of the tested price range. The baseline assigns 10 to never-buyers as a simplifying convention, not an observed reservation price; consider its implications when interpreting the results.


*Your answer:*



## Similarity / distance metrics

We need a notion of 'closeness' between observations. We'll start with numeric variables only to keep it simple, then add categorical variables.

### A warm-up: Distances between two households

We'll pick two random observations and compute distances under different metrics.

In [ ]:
# Pick two observations to compare
i, j = np.random.choice(len(X), size=2, replace=False)
row_i = X.iloc[i]
row_j = X.iloc[j]
row_i, row_j


In [ ]:
# We'll compute distances using numeric columns only (for now)
xi = row_i[num_cols].to_numpy().astype(float)
xj = row_j[num_cols].to_numpy().astype(float)

def l2(a, b):
    """Euclidean distance: straight-line distance between two points."""
    return float(np.sqrt(np.sum((a - b)**2)))

def l1(a, b):
    """Manhattan distance: total absolute difference across features."""
    return float(np.sum(np.abs(a - b)))

def linf(a, b):
    """Max (L-infinity) distance: largest difference on any single feature."""
    return float(np.max(np.abs(a - b)))

print("L2 (Euclidean):", l2(xi, xj))
print("L1 (Manhattan):", l1(xi, xj))
print("L∞ (Sup):", linf(xi, xj))


---

**Question 2.** Which variable(s) dominate these distances right now? Why?
What will happen to these distances if we standardize numeric variables first?

---

*Your answer:*



### A baseline representation

Many unsupervised methods are sensitive to scale and to the way categorical variables are encoded.  As a simple default, we will:

1. standardize numeric variables,
2. one-hot encode categorical variables, and
3. use that representation for both visualization and clustering.

This is a useful baseline, but it is not automatically the "right" notion of similarity for the economic question.


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop",
)

X_rep = preprocessor.fit_transform(X)

print("Representation shape:", X_rep.shape)


---

**Question 3.** Why might standardization matter especially strongly in a clustering problem?  More broadly, does the representation above make an implicit judgment about what it means for two households to be "similar"?  Briefly explain.

Think about the variable `state`. After one-hot encoding, what are we implicitly saying about distances between states? Suggest an alternative way to capture the categorical variable `state`. (HINT: You can think about augmenting the data.) More generally, comment on the pros and cons of the practice of one-hot encoding categorical variables for use in clustering.

---

*Your answer:*




## Baseline clustering: K-means

We now build a simple segmentation using K-means.  As in the other cases, the point is to create a baseline result that you can critique and extend.

We fit K-means for several values of `K` and inspect two common diagnostics:

- **Inertia**: lower means tighter clusters.
- **Silhouette score**: higher means better separation.


In [ ]:
def fit_kmeans_grid(Xmat, k_grid, random_state=42, n_init=10):
    rows = []
    models = {}
    for k in k_grid:
        km = KMeans(n_clusters=k, random_state=random_state, n_init=n_init)
        labels = km.fit_predict(Xmat)
        rows.append({
            "K": k,
            "inertia": km.inertia_,
            "silhouette": silhouette_score(Xmat, labels)
        })
        models[k] = km
    return pd.DataFrame(rows), models

k_grid = list(range(2, 9))
km_table, km_models = fit_kmeans_grid(X_rep, k_grid)
km_table


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(km_table["K"], km_table["inertia"], marker="o")
ax[0].set_title("Elbow plot")
ax[0].set_xlabel("K")
ax[0].set_ylabel("Inertia")
ax[0].grid(True)

ax[1].plot(km_table["K"], km_table["silhouette"], marker="o")
ax[1].set_title("Silhouette score")
ax[1].set_xlabel("K")
ax[1].set_ylabel("Silhouette")
ax[1].grid(True)

plt.show()


**Question 4.** Choose a baseline value of `K` using the diagnostics. Explain why the choice also depends on the economic question and interpretability, rather than following mechanically from one numerical criterion.


*Your answer:*



## Inspecting one baseline segmentation

The next cell fixes a baseline number of clusters and produces simple summaries.  You should feel free to change `K_BASELINE`.

In [ ]:
K_BASELINE = 3

km = km_models[K_BASELINE]
labels = km.labels_

profile = households.copy()
profile["cluster"] = labels

print("Cluster sizes:")
print(profile["cluster"].value_counts().sort_index())

cluster_means = profile.groupby("cluster")[[c for c in num_cols if c in profile.columns]].mean()
cluster_means

**Question 5.** Give each cluster an economically interpretable description based on its profile. Compare household characteristics and observed purchase responses across clusters. Distinguish evidence of different willingness-to-pay levels from evidence about price sensitivity.


*Your answer:*



### Extension: Representation and Robustness

**Question 6.** Extend the household analysis:

1. Choose a representation of households and explain why it is appropriate for studying heterogeneity in purchase responses.
2. Go beyond the exact baseline using an alternative clustering method, a different encoding of similarity, PCA, or another suitable unsupervised method.
3. Assess whether your conclusions are stable when you vary feature construction, `K`, or the method.

Explain the method's objective, what constitutes informative evidence of household heterogeneity, and which additional data would help validate your interpretation. If purchase-derived features are used to form clusters, differences in those same features across clusters are not independent validation.


*Your answer:*



Congrats, you finished the first homework! :)

Please submit your solution to the TA (paul.stephan@econ.uzh.ch), one submission per group is enough. Please make sure to indicate the names of all group members in the email. Thanks!